# 02. Model Training and Evaluation
This notebook trains classification models (Logistic Regression, Random Forest, XGBoost) to predict the outcome of World Cup matches.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
import shap
import pickle
import matplotlib.pyplot as plt

# Load Data
df = pd.read_csv('../data/wc_training_dataset_updated.csv')

FEATURE_COLS = [
    'home_fifa_rank', 'home_fifa_points', 'home_wc_win_rate',
    'home_avg_goals_scored', 'home_avg_goals_conceded', 'home_goal_diff',
    'home_wc_matches', 'home_wc_appearances', 'home_wc_titles',
    'away_fifa_rank', 'away_fifa_points', 'away_wc_win_rate',
    'away_avg_goals_scored', 'away_avg_goals_conceded', 'away_goal_diff',
    'away_wc_matches', 'away_wc_appearances', 'away_wc_titles',
    'rank_diff', 'points_diff', 'points_ratio',
    'win_rate_diff', 'goal_diff_diff', 'title_diff', 'experience_diff'
]

X = df[FEATURE_COLS]
y = df['result']   # H, D, A

# Time-based Train/Test Split
X_train = X[df['year'] <= 2014]
y_train = y[df['year'] <= 2014]
X_test  = X[df['year'] == 2018]
y_test  = y[df['year'] == 2018]

print(f"Train set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Train set shape: (372, 25)
Test set shape: (64, 25)


### Train Models and Evaluate

In [2]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05,
                                         max_depth=4, random_state=42,
                                         eval_metric='mlogloss')
}

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

results = {}
for name, model in models.items():
    if name == 'XGBoost':
        model.fit(X_train, y_train_encoded)
        preds = model.predict(X_test)
        preds_decoded = le.inverse_transform(preds)
        proba = model.predict_proba(X_test)
    else:
        model.fit(X_train, y_train)
        preds_decoded = model.predict(X_test)
        proba = model.predict_proba(X_test)
    
    acc = accuracy_score(y_test, preds_decoded)
    ll = log_loss(y_test_encoded, proba)
    results[name] = {'model': model, 'accuracy': acc, 'log_loss': ll, 'proba': proba, 'preds': preds_decoded}
    print(f"{name}: Accuracy={acc:.3f}, Log Loss={ll:.3f}")

Logistic Regression: Accuracy=0.469, Log Loss=1.067
Random Forest: Accuracy=0.469, Log Loss=1.120
XGBoost: Accuracy=0.438, Log Loss=1.324


C:\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Select Best Model

In [3]:
best_name  = min(results, key=lambda k: results[k]['log_loss'])
best_model = results[best_name]['model']
print(f"Best model by Log Loss: {best_name}")

import os
os.makedirs('../models', exist_ok=True)
with open('../models/xgb_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("Best model saved to models/xgb_model.pkl")

Best model by Log Loss: Logistic Regression
Best model saved to models/xgb_model.pkl


### Baseline Comparison

In [4]:
# Baseline 1: always predict Home Win
baseline_acc = (y_test == 'H').mean()
print(f"Baseline (always Home Win) accuracy: {baseline_acc:.3f}")

# Baseline 2: predict winner based on FIFA rank alone
rank_preds = ['H' if row['rank_diff'] < 0 else 'A' for _, row in X_test.iterrows()]
rank_acc   = accuracy_score(y_test, rank_preds)
print(f"Baseline (FIFA rank only) accuracy: {rank_acc:.3f}")

Baseline (always Home Win) accuracy: 0.391
Baseline (FIFA rank only) accuracy: 0.500


### Calibration Plot

In [5]:
classes = list(best_model.classes_) if hasattr(best_model, 'classes_') else ['A', 'D', 'H']
h_idx = classes.index('H')

prob_true, prob_pred = calibration_curve(
    (y_test == 'H').astype(int),
    results[best_name]['proba'][:, h_idx],
    n_bins=5
)
plt.figure(figsize=(6, 4))
plt.plot(prob_pred, prob_true, marker='o', label='Model')
plt.plot([0,1],[0,1], linestyle='--', label='Perfect calibration')
plt.xlabel('Predicted probability')
plt.ylabel('Actual win rate')
plt.title('Calibration Plot — Home Win')
plt.legend()
plt.savefig('../calibration_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### Confusion Matrix

In [6]:
cm = confusion_matrix(y_test, results[best_name]['preds'], labels=classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}')
plt.savefig('../confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### SHAP Explainer Setup (for XGBoost)

In [7]:
xgb_model = results['XGBoost']['model']
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

with open('../models/shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)
print("SHAP explainer saved to models/shap_explainer.pkl")

SHAP explainer saved to models/shap_explainer.pkl
